# Stage 2 Data Prep — Physical Access

This notebook verifies and prepares the Stage 2 (can_access_in_person) data before it's used in voter_access.ipynb, the final model notebook.

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

import pandas as pd

stage2_access.csv is manually transcribed from Census Table 10 (not an automated export), due to the source file's multi-header formatting. See SOURCES.md for full methodology and the population caveat.

In [15]:
stage2 = pd.read_csv("data/stage2_access.csv")
stage2

,measure,subgroup,percentage,population_count,denominator,source_key,confidence_tier,notes
0,reason_not_voting,too_busy_conflicting_schedule,17.8,3232700,registered_but_did_not_vote,cps_2024_table10,published,Closest proxy for 'can't physically get there'...
1,reason_not_voting,transportation_problems,2.2,399500,registered_but_did_not_vote,cps_2024_table10,published,Most direct proxy for physical access barrier
2,reason_not_voting,illness_or_disability,12.4,2252000,registered_but_did_not_vote,cps_2024_table10,published,"Physical access barrier, different mechanism t..."
3,reason_not_voting,out_of_town,7.4,1343900,registered_but_did_not_vote,cps_2024_table10,published,Access barrier - not physically present
4,reason_not_voting,registration_problems,3.6,653800,registered_but_did_not_vote,cps_2024_table10,published,"Direct hit on our registration-gate concept, t..."
5,reason_not_voting,inconvenient_polling_place,2.4,435900,registered_but_did_not_vote,cps_2024_table10,published,"Access barrier, polling-location specific"
6,reason_not_voting,not_interested,19.7,3577700,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
7,reason_not_voting,forgot_to_vote,4.1,744600,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
8,reason_not_voting,didnt_like_candidates_or_campaign,14.7,2669700,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...
9,reason_not_voting,bad_weather,0.3,54500,registered_but_did_not_vote,cps_2024_table10,published,Not an access barrier - included only for the ...


# Verification

Each row was manually cross-referenced against the original Census Table 10 during transcription; the sum-to-100% check below is the objective verification of that process.

In [22]:
full_sum = round(stage2["percentage"].sum(),1)
print(f"Sum of all 12 reason percentages: {full_sum}%")

Sum of all 12 reason percentages: 99.9%


# Cross Checking

sanity-check the data in two different way: 
1) using the total citizen population implied by our Stage 1 numbers, plus the baseline registration/turnout rates, we can estimate how many people should be "registered but didn't vote" 
2) compare the above estimate to what CPS Table 10 actually reports. If our two independent sources roughly agree, that's a good sign our numbers are trustworthy.

In [19]:
# Baseline turnout/registration figures, from the Census press release
# (2024 Presidential Election Voting and Registration Tables)
pct_registered = 0.736
pct_voted = 0.653

In [20]:
# total voting-age citizen population from Stage 1 data
lacks_access_count = 21_300_000
lacks_access_pct = 0.091
total_voting_age_citizens = lacks_access_count / lacks_access_pct

# using baseline rates to estimate how many people should be registered but did not vote
estimated_registered = total_voting_age_citizens * pct_registered
estimated_voted = total_voting_age_citizens * pct_voted
estimated_registered_not_voted = estimated_registered - estimated_voted

# Comparing estimate with CPS table
actual_registered_not_voted = 18_161_000
difference = abs(estimated_registered_not_voted - actual_registered_not_voted)
pct_difference = (difference / actual_registered_not_voted) * 100

print(f"Estimated total voting-age citizens: {total_voting_age_citizens:,.0f}")
print(f"Estimated registered-but-didn't-vote: {estimated_registered_not_voted:,.0f}")
print(f"CPS Table 10's actual figure: {actual_registered_not_voted:,.0f}")
print(f"Difference: {pct_difference:.1f}%")


Estimated total voting-age citizens: 234,065,934
Estimated registered-but-didn't-vote: 19,427,473
CPS Table 10's actual figure: 18,161,000
Difference: 7.0%


our estimate of 19.4M and the CPS table's actual figure of 18.16M are within 7% of each other. Given that these numbers come from two different surveys: one from in 2023 (the DPOC survey) and one from 2024 (CPS), with different sample designs, this level of agreement is a good sign the numbers are consistent with each other. 

# Stage 2 access-barrier rate (combined proxy)

Rather than isolating a single reason (e.g., transportation problems alone), we sum all six access-related reasons from CPS Table 10: illness/disability, out of town, too busy/conflicting schedule, transportation problems, registration problems, and inconvenient polling place. This gives a broader "couldn't make it in person, regardless of the specific cause" rate, which fits our model's goal of identifying which stage is the bottleneck rather than diagnosing the specific socioeconomic driver within that stage — a more granular breakdown (e.g., by income) would be a natural extension with more time. Summing is valid here because CPS Table 10 is a single-select question — each respondent reported one primary reason, so these categories don't overlap.


In [14]:
access_barrier_reasons = [
    "too_busy_conflicting_schedule",
    "transportation_problems",
    "illness_or_disability",
    "out_of_town",
    "registration_problems",
    "inconvenient_polling_place",
]

stage2_filtered = stage2[stage2["subgroup"].isin(access_barrier_reasons)]
access_barrier_rate = stage2_filtered["percentage"].sum()

print(f"Combined access-barrier rate: {access_barrier_rate}%")

Combined access-barrier rate: 45.8%


In [26]:
p_no_access = round(access_barrier_rate / 100,3)
p_has_access = round(1 - p_no_access,3)

print(f"P(can_access_in_person = No): {p_no_access}")
print(f"P(can_access_in_person = Yes): {p_has_access}")

P(can_access_in_person = No): 0.458
P(can_access_in_person = Yes): 0.542


# Limitations

- The population isn't a perfect match: 
    This table only includes people who are already registered but didn't end up voting (18.16M people) — not everyone who might have gotten blocked earlier, like at registration. We're using it as a stand-in for "physical access barriers" under the assumption that the same things (transportation, time) would also make it hard for someone to register in person — but that's an assumption we're making, not something the data directly proves.

- We didn't break this down by income or other groups:
    The CPS table does have some breakdowns (like by age or income) for a few of the reasons, but we're just using the flat national number for time's sake. Splitting it out more, like showing a higher barrier rate for lower-income people, would be a good next step if we had more time.

- We combined six different reasons into one number:
     Instead of looking at each reason separately (illness, transportation, being too busy, etc.), we added them together into one "couldn't make it" rate. This is fine to do since each person only picked one reason (so nothing's being double-counted), but it does mean we can't say which specific reason matters most — just that 45.8% of people ran into some kind of access problem.

# Stage 2 prep is done
This gives P(can_access_in_person = No) = 0.458 for use in voter_access.ipynb's CPTs